# Scraper et crawler un vrai site : Books to Scrape
**Un site réel, public, créé exactement pour s'entraîner** : `books.toscrape.com` (une librairie fictive de 1 000 livres, sur 50 pages).

Dans ce notebook, vous écrivez un script **classique** : `requests` pour télécharger, `BeautifulSoup` pour lire, des boucles pour parcourir. Rien d'autre. À la fin, vous aurez un fichier CSV de livres.

| Étape | Ce que vous faites |
|---|---|
| 1 | Télécharger une page et la lire |
| 2 | Repérer un livre dans le HTML |
| 3 | Extraire ses champs |
| 4 | Faire la liste des livres d'une page |
| 5 | **Crawler** : suivre le bouton « next » de page en page |
| 6 | **Aller plus loin** : ouvrir la fiche de chaque livre |
| 7 | Sauvegarder et contrôler |

**Pour l'instant, on ne se pose aucune question de politesse ni de règles** : ce site est fait pour ça. Le notebook suivant expliquera pourquoi, sur d'autres sites, il faut faire autrement.

`pip install requests beautifulsoup4 pandas`

In [58]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

URL_DEPART = "https://books.toscrape.com/"
def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Télécharger une page et la lire
`requests.get` télécharge la page. Le **code 200** veut dire « tout va bien ». `BeautifulSoup` transforme le texte HTML en un arbre dans lequel on peut chercher.

💡 On donne à BeautifulSoup `reponse.content` (les octets bruts) et non `reponse.text` : il devine alors lui-même l'encodage correct. Sinon, le symbole `£` peut s'afficher `Â£`.

In [59]:
reponse = requests.get(URL_DEPART)
print("Code de réponse :", reponse.status_code)

page = BeautifulSoup(reponse.content, "html.parser")
print("Titre de la page :", page.title.get_text(strip=True))

Code de réponse : 200
Titre de la page : All products | Books to Scrape - Sandbox


## Étape 2 — Repérer un livre dans le HTML
Ouvrez le site dans votre navigateur, faites un clic droit sur un livre puis **Inspecter** : vous verrez que chaque livre est dans une balise `<article class="product_pod">`. Regardons-en un.

In [60]:
carte = page.select_one("article.product_pod")
print(carte.prettify()[:900])

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



## Étape 3 — Extraire les champs d'un livre
On écrit une fonction qui lit **une carte** et renvoie un dictionnaire. Trois choses à noter :
- le titre complet est dans l'attribut `title` du lien (le texte affiché est parfois coupé) ;
- la note est dans le **nom de la classe** (`star-rating Three`), pas dans le texte ;
- les liens sont **relatifs** : `urljoin` fabrique l'adresse complète.

In [61]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def lire_livre(carte, url_page):
    lien = carte.select_one("h3 a")
    prix_texte = carte.select_one("p.price_color").get_text(strip=True)
    return {
        "titre": lien["title"],
        "prix": float(re.sub(r"[^0-9.]", "", prix_texte)),          # ne garde que les chiffres et le point
        "note": NOTES[carte.select_one("p.star-rating")["class"][1]],
        "url": urljoin(url_page, lien["href"]),
    }

print(lire_livre(carte, URL_DEPART))

{'titre': 'A Light in the Attic', 'prix': 51.77, 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


## Étape 4 — Tous les livres d'une page

In [62]:
livres_page = [lire_livre(c, URL_DEPART) for c in page.select("article.product_pod")]
print(len(livres_page), "livres sur la première page")
pd.DataFrame(livres_page).head()

20 livres sur la première page


,titre,prix,note,url
0,A Light in the Attic,51.77,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,https://books.toscrape.com/catalogue/sapiens-a...


## Étape 5 — Crawler : suivre le bouton « next »
C'est le **crawling** : on part d'une page, on lit ses livres, puis on suit le lien vers la page suivante, jusqu'à ce qu'il n'y en ait plus. La boucle s'arrête toute seule quand il n'y a plus de bouton « next ».

In [63]:
livres = []
url = URL_DEPART
numero = 0
while url:
    numero += 1
    page = BeautifulSoup(requests.get(url).content, "html.parser")
    for carte in page.select("article.product_pod"):
        livres.append(lire_livre(carte, url))
    suivant = page.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None
    print(f"page {numero} lue, {len(livres)} livres au total")

print("\nTotal :", len(livres), "livres sur", numero, "pages")

page 1 lue, 20 livres au total
page 2 lue, 40 livres au total
page 3 lue, 60 livres au total
page 4 lue, 80 livres au total
page 5 lue, 100 livres au total
page 6 lue, 120 livres au total
page 7 lue, 140 livres au total
page 8 lue, 160 livres au total
page 9 lue, 180 livres au total
page 10 lue, 200 livres au total
page 11 lue, 220 livres au total
page 12 lue, 240 livres au total
page 13 lue, 260 livres au total
page 14 lue, 280 livres au total
page 15 lue, 300 livres au total
page 16 lue, 320 livres au total
page 17 lue, 340 livres au total
page 18 lue, 360 livres au total
page 19 lue, 380 livres au total
page 20 lue, 400 livres au total
page 21 lue, 420 livres au total
page 22 lue, 440 livres au total
page 23 lue, 460 livres au total
page 24 lue, 480 livres au total
page 25 lue, 500 livres au total
page 26 lue, 520 livres au total
page 27 lue, 540 livres au total
page 28 lue, 560 livres au total
page 29 lue, 580 livres au total
page 30 lue, 600 livres au total
page 31 lue, 620 livres

In [64]:
controle(len(livres) > 0 and len({l["url"] for l in livres}) == len(livres),
         "%d livres, sans doublon d'adresse (sur le vrai site, attendez-vous à 1000 livres sur 50 pages)" % len(livres),
         "Attendu : au moins un livre, et aucune adresse en double")

✅ 1000 livres, sans doublon d'adresse (sur le vrai site, attendez-vous à 1000 livres sur 50 pages)


## Étape 6 — Aller plus loin : ouvrir la fiche de chaque livre
La page de liste ne donne ni la **catégorie**, ni la **description**, ni le **stock**. Il faut ouvrir la fiche de chaque livre : c'est du crawling vers des pages de détail.

⚠️ Attention : **certains livres n'ont pas de description**. Le code doit le prévoir (`None`), sinon il plante au premier livre concerné.

⏱️ Ouvrir 1 000 fiches prend plusieurs minutes. On commence avec `LIMITE = 40`, puis on met `None` pour tout lire.

In [65]:
def lire_detail(url):
    page = BeautifulSoup(requests.get(url).content, "html.parser")
    liens_fil = page.select("ul.breadcrumb li a")                     # Home, Books, <catégorie>
    description = page.select_one("#product_description + p")        # absente sur certains livres
    infos = {r.th.get_text(strip=True): r.td.get_text(strip=True) for r in page.select("table.table-striped tr")}
    stock = re.search(r"\d+", infos.get("Availability", ""))
    return {
        "categorie": liens_fil[2].get_text(strip=True) if len(liens_fil) >= 3 else None,
        "description": description.get_text(strip=True) if description else None,
        "upc": infos.get("UPC"),
        "stock": int(stock.group()) if stock else None,
        "nb_avis": int(infos.get("Number of reviews", 0)),
    }

LIMITE = 40            # mettez None pour lire toutes les fiches
details = []
for i, livre in enumerate(livres[:LIMITE]):
    details.append({**livre, **lire_detail(livre["url"])})
    if (i + 1) % 10 == 0:
        print(i + 1, "fiches lues")
df = pd.DataFrame(details)
df.head()

10 fiches lues
20 fiches lues
30 fiches lues
40 fiches lues


,titre,prix,note,url,categorie,description,upc,stock,nb_avis
0,A Light in the Attic,51.77,3,https://books.toscrape.com/catalogue/a-light-i...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,22,0
1,Tipping the Velvet,53.74,1,https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,20,0
2,Soumission,50.10,1,https://books.toscrape.com/catalogue/soumissio...,Fiction,"Dans une France assez proche de la nôtre, un h...",6957f44c3847a760,20,0
3,Sharp Objects,47.82,4,https://books.toscrape.com/catalogue/sharp-obj...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,20,0
4,Sapiens: A Brief History of Humankind,54.23,5,https://books.toscrape.com/catalogue/sapiens-a...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,20,0


## Étape 7 — Sauvegarder et contrôler
Un fichier de données se **vérifie** avant de servir : lignes, doublons, valeurs manquantes.

In [66]:
df.to_csv("livres.csv", index=False)
print("Lignes :", len(df), "| doublons d'UPC :", df["upc"].duplicated().sum())
print("\nValeurs manquantes par colonne :")
print(df.isna().sum().to_string())
print("\nLivres par catégorie :")
print(df["categorie"].value_counts().head(8).to_string())

Lignes : 40 | doublons d'UPC : 0

Valeurs manquantes par colonne :
titre          0
prix           0
note           0
url            0
categorie      0
description    0
upc            0
stock          0
nb_avis        0

Livres par catégorie :
categorie
Default        7
Poetry         5
Music          3
Thriller       3
Mystery        2
Young Adult    2
Romance        2
Childrens      2


In [67]:
controle(len(df) > 0 and df["upc"].notna().all() and df["prix"].between(0, 1000).all(),
         "Fichier livres.csv cohérent : %d lignes, prix plausibles" % len(df),
         "Attendu : des lignes, un UPC par livre et des prix entre 0 et 1000")

✅ Fichier livres.csv cohérent : 40 lignes, prix plausibles


## À vous
Sans copier de solution, complétez dans de nouvelles cellules :
1. Quelle est la **note moyenne** par catégorie ? Et le **prix moyen** par note ?
2. **Combien de livres n'ont pas de description** sur ceux que vous avez lus ?
3. Ajoutez au dictionnaire un champ **« disponible »** (vrai si le stock est supérieur à 0).
4. Faites tourner la collecte sur **une seule catégorie** : trouvez comment, dans le menu de gauche du site, atteindre la page d'une catégorie, et adaptez votre boucle.

## Questions
- Combien de requêtes votre script a-t-il envoyées en tout pour 1 000 livres (pages de liste + fiches) ? Combien de temps a-t-il pris ?
- Qu'est-ce qui plante votre script si le site change le nom d'une classe ?
- Votre script envoie ses requêtes **l'une après l'autre, aussi vite que possible**, sans rien dire de lui. Que se passerait-il, à votre avis, sur un site qui n'est pas fait pour ça ?

In [83]:
#pour savoir quelles colonnes sont présentes dans le DataFrame
display(df.columns)

Index(['titre', 'prix', 'note', 'url', 'categorie', 'description', 'upc',
       'stock', 'nb_avis'],
      dtype='str')

In [76]:
# 1 Moyenne note et prix

def inventaires_livres(df):
    note_moyenne = df['note'].mean()
    prix_moyen = df['prix'].mean().round(2)
    livres_sans_description = df[df['description'].isnull()]
    
    return note_moyenne, prix_moyen, livres_sans_description

note_moyenne, prix_moyen, livres_sans_description = inventaires_livres(df)
print(f"Note moyenne : {note_moyenne}")
print(f"Prix moyen : {prix_moyen}")
# livres sans description
print(f"Nombre de livres sans description : {len(livres_sans_description)}")

Note moyenne : 3.05
Prix moyen : 34.96
Nombre de livres sans description : 0


In [82]:
# Création de dictionnaires de livres en stock > 0

def livres_en_stock(df):
    livres_stock = df[df['stock'] > 0]
    dictionnaire_livres = livres_stock.to_dict(orient='records')
    return dictionnaire_livres
livres_stock = livres_en_stock(df)
print(f"Nombre de livres en stock : {len(livres_stock)}")

Nombre de livres en stock : 40


In [81]:
# tourner la collecte sur une seule catégorie, "Travel"

accueil = BeautifulSoup(requests.get(URL_DEPART).content, "html.parser")
cats = {a.text.strip(): urljoin(URL_DEPART, a["href"])
        for a in accueil.select("div.side_categories ul li ul li a")}
print(cats["Travel"])   # doit contenir /catalogue/category/books/travel_2/

https://books.toscrape.com/catalogue/category/books/travel_2/index.html
